In [1]:
import pandas as pd
import numpy as np
from anjana.anonymity import (
    k_anonymity_inner,
    k_anonymity,
    l_diversity,
    t_closeness,
    alpha_k_anonymity,
)
import masking_effects_on_xai_techniques.hierarchies as hi

In [2]:
path = "../data/usa_house/"

In [3]:
data = pd.read_csv(path + "data.csv")
len_data = len(data)

# Clean the data
data.dropna(inplace=True)
data = data.drop("Address", axis=1)

len_clean_data = len(data)
print(f"Dropped {len_data - len_clean_data} rows")
df = data
data.to_csv(path + "clean.csv", index=False)

Dropped 0 rows


In [4]:
data.nunique()

Avg. Area Income                5000
Avg. Area House Age             5000
Avg. Area Number of Rooms       5000
Avg. Area Number of Bedrooms     255
Area Population                 5000
Price                           5000
dtype: int64

All features are continous numbers, except the Avg. Area Number of Bedrooms. Therefore we set level of bins at 50 for all features except that single one. We set that at 5. Then there are approxmiately 50 elements in the first level

In [5]:
data.columns

Index(['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms',
       'Avg. Area Number of Bedrooms', 'Area Population', 'Price'],
      dtype='object')

In [27]:
def create_and_save(feat, n):
    hierarchies[feat] = hi.generate_qcut_hierarchy(data[feat], n)
    hi.save_hierarchy(hierarchies[feat], f"../hierarchies/usa_house/{feat}.csv")


categories = data.columns.to_list()
bedroom = "Avg. Area Number of Bedrooms"
categories.remove(bedroom)
categories.remove("Price")

hierarchies = dict()
for cat in categories:
    create_and_save(cat, 3)
    hierarchies[cat] = dict(
        pd.read_csv(f"../hierarchies/usa_house/{cat}.csv", header=None)
    )

create_and_save(bedroom, 3)
hierarchies[bedroom] = dict(
    pd.read_csv(f"../hierarchies/usa_house/{bedroom}.csv", header=None)
)

/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/pandas/core/indexes/base.py:7819: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(values).astype(str)
/home/kochc/projects/bachelor/Masking-Effects-on-XAI-Techniques/.venv/lib/python3.12/site-packages/p

In [7]:
ident = []
sens_att = "Price"
quasi_ident = list(data.columns)
quasi_ident.remove(sens_att)

In [8]:
all_counts_and_features = []
for feat, items in hierarchies.items():
    for item in items.values():
        length = len(item.unique())
        all_counts_and_features.append((length, feat))
sorted_data = sorted(all_counts_and_features, key=lambda x: x[0], reverse=True)
ordered_features = [feat for count, feat in sorted_data]

for feat, items in hierarchies.items():
    lengths = [f"{len(item.unique()):>4}" for item in items.values()]
    out = ", ".join(lengths)
    print(f"{feat:<15}: {out}")

print("Order in which features will be generalized:")
print(ordered_features)

Avg. Area Income: 5000,   20,   19,   18,   17,   16,   15,   14,   13,   12,   11,   10,    9,    8,    7,    6,    5,    4,    3,    2,    1
Avg. Area House Age: 5000,   20,   19,   18,   17,   16,   15,   14,   13,   12,   11,   10,    9,    8,    7,    6,    5,    4,    3,    2,    1
Avg. Area Number of Rooms: 5000,   20,   19,   18,   17,   16,   15,   14,   13,   12,   11,   10,    9,    8,    7,    6,    5,    4,    3,    2,    1
Area Population: 5000,   20,   19,   18,   17,   16,   15,   14,   13,   12,   11,   10,    9,    8,    7,    6,    5,    4,    3,    2,    1
Avg. Area Number of Bedrooms:  255,    5,    4,    3,    2,    1
Order in which features will be generalized:
['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms', 'Area Population', 'Avg. Area Number of Bedrooms', 'Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms', 'Area Population', 'Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms', 'Area Population', '

In [28]:
%%time
k = 10
supp_level = 20
anon_df, supp_n, hiear = k_anonymity_inner(
    data, ident, quasi_ident, k, supp_level, hierarchies
)
max_supp_n = int(round(len(data) * supp_level / 100, 0))
print(f"Max rows that can be suppressed: {max_supp_n}")
print(f"Rows suppressed                : {supp_n}")
print(f"% of allowed rows suppressed   : {round(supp_n / max_supp_n * 100, 1)}%")
print(f"Generalization level           : {sum(hiear.values())}")
hiear

Max rows that can be suppressed: 1000
Rows suppressed                : 222
% of allowed rows suppressed   : 22.2%
Generalization level           : 5
CPU times: user 1.14 s, sys: 172 ms, total: 1.31 s
Wall time: 1.31 s


{'Avg. Area Income': 1,
 'Avg. Area House Age': 1,
 'Avg. Area Number of Rooms': 1,
 'Avg. Area Number of Bedrooms': 1,
 'Area Population': 1}

In [ ]:
k = 10
supp_level = 20
anon_df, supp_n, hiear = k_anonymity_inner(
    data, ident, quasi_ident, k, supp_level, hierarchies
)
max_supp_n = int(round(len(data) * supp_level / 100, 0))
print(f"Max rows that can be suppressed: {max_supp_n}")
print(f"Rows suppressed                : {supp_n}")
print(f"% of allowed rows suppressed   : {round(supp_n / max_supp_n * 100, 1)}%")
print(f"Generalization level           : {sum(hiear.values())}")
hiear

Why prefer the one over the other? Idk  
The one with lower levels suppres fewer rolls.  
The one with lower levels generalize all the way down to the level where there are 25-33% elements in every bin  
The one with higher levels only generalize to having 20% of elements in each bin  
How to get these numbers? Do `max-level - gen_level + 1`. That is the amount of bins created.  
In conclusion, They are quite close to one another, the one with fewer levels just supresses more rows for some reason. This is reason why it is also generalized more, since there are more instances that can be part of the same equivalence classes